# 02 – Join Layers to SWORD
**Purpose:** Expand SWORD reaches with attributes from vector and raster dataset. Each section adds columns to a growing SWORD GeoDataFrame. The final output contains all joined attributes.

**Workflow:**

2.0 | Imports

2.1 | Vector Data

- 2.1.1 | Line Join
- 2.1.2 | Point Join
- 2.1.3 | Polygon

2.2 | Raster Data
- 2.2.1 | STI glowabio

To-Do:
- [x] erstellte outputs aufeinander aufbauen lassen, sodass der SWORD Datensatz immer erweitert wird.
- [ ] Muss noch im notebook den SWORD datensatzaufbau einbauen und übersichtlicher gestalten
- [ ] sections einfügen, um zum SWORD hinzugefügte Datensätze mit dem original zu vergleichen

---
---
## 2.0 | Imports

In [1]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import sys
import fiona
import matplotlib.pyplot as plt
import contextily as ctx
import webbrowser

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT
from _2_1_1_line import join_line_majority
from _2_1_2_point import snap_points_to_lines
from _2_2_1_raster import extract_raster_values

#### Input / Output cell:

In [2]:
# ============================================================
# INPUT / OUTPUT OVERVIEW
# ============================================================

#--- Base input (from Notebook 01) ---------------------------
IN_SWORD = os.path.join(DATA_PROCESSED, "sword_naryn_clip.gpkg")

#--- 2.1.1 Line ----------------------------------------------
IN_GRIT = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_GRITv1.0.gpkg")
GRIT_LAYER = None

OUT_2_1_1 = os.path.join(DATA_PROCESSED, "sword_naryn_2_1_1_grit.gpkg")
OUT_MAP_2_1_1 = os.path.join(DATA_OUTPUT, "02_1_1_grit_map.html")

#--- 2.1.2 Point ---------------------------------------------
IN_GDW = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_GDW_barriers_v1_0.gpkg")
OUT_2_1_2 = os.path.join(DATA_PROCESSED, "sword_naryn_2_1_2_gdw_snapped.gpkg")

#--- 2.1.3 Polygon Join: ... -------------------------


#--- 2.2.1 Raster Join: ---------------------
RASTER_JOINS = [
    ("STI_glowabio", os.path.join(DATA_RAW, "0_data", "Naryn_example", "glowabio", "Naryn_sti.tif")),
    # ("next_raster", os.path.join(DATA_RAW, "...")),
]
OUT_2_2_1 = os.path.join(DATA_PROCESSED, "sword_naryn_2_2_1_raster.gpkg")

#--- Final outputs --------------------------------------------------
OUT_FINAL     = os.path.join(DATA_PROCESSED, "sword_naryn_joined_final.gpkg")
OUT_MAP_FINAL = os.path.join(DATA_OUTPUT,    "02_final_map.html")

In [3]:
# Vertify all inputs exists:

check = [
    ("SWORD clipped", IN_SWORD),
    ("GRIT",IN_GRIT),
    ("GDW",IN_GDW),
] + [(f"Raster: {name}", path) for name, path in RASTER_JOINS]

all_ok = True
for label, path in check:
    exists = os.path.exists(path)
    status = "Found" if exists else "NOT FOUND: check path"
    print(f"{label:30s}: {status}")
    if not exists:
        all_ok = False

print()
print("All files found." if all_ok else "Fix missing paths.")

SWORD clipped                 : Found
GRIT                          : Found
GDW                           : Found
Raster: STI_glowabio          : Found

All files found.


---
---
## 2.1 | Vector Data

---
### 2.1.1 | Line Join – GRIT strahler order

Joins GRITv1.0 strahler order to each SWORD reach using majority-vote sampling. Sample points are placed every 50m along each reach. Each point is assigned to the nearest GRIT feature. The strahler order with the most votes wins.

To-Do:
- [ ] Überprüfen, ob mehrheitsvote nach wie vor funktioniert
- [ ] How should I handle reaches which falls within the coverage of multiple strahler?


<u>References:</u>

References:

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025) URL: https://aquaknow.jrc.ec.europa.eu/dataset/global-river-topology-grit

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025). Global River Topology (GRIT): A bifurcating river hydrography. Water Resources Research, 61, e2024WR038308. https://doi.org/10.1029/2024WR038308


In [7]:
# Load Data:
sword = gpd.read_file(IN_SWORD)
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

grit_layers = fiona.listlayers(IN_GRIT)
grit_layer = GRIT_LAYER if GRIT_LAYER else grit_layers[0]
grit_raw = gpd.read_file(IN_GRIT, layer=grit_layer)
print(f"GRIT loaded: {len(grit_raw)} features | CRS: {grit_raw.crs}")
print(f"GRIT columns: {grit_raw.columns.tolist()}")

SWORD loaded: 140 reaches | CRS: EPSG:4326
GRIT loaded: 710 features | CRS: EPSG:4326
GRIT columns: ['cat', 'global_id', 'catchment_id', 'upstream_node_id', 'downstream_node_id', 'upstream_line_ids', 'downstream_line_ids', 'direction_algorithm', 'width_adjusted', 'length_adjusted', 'is_mainstem', 'strahler_order', 'cycle', 'length', 'azimuth', 'sinuousity', 'drainage_area_in', 'drainage_area_out', 'drainage_area_mainstem_in', 'drainage_area_mainstem_out', 'bifurcation_balance_out', 'grwl_overlap', 'grwl_value', 'name', 'name_local', 'n_bifurcations_upstream', 'domain', 'geometry']


Configuration Cell:

In [8]:
# Configuration:
# Which cols should be joined
GRIT_COLS_TO_JOIN = ["strahler_order", "global_id", "geometry"]
# Rename added columns in SWORD dataset
GRIT_RENAME= {
    "strahler_order" : "strahler_order_GRITv1.0",
    "global_id" : "global_id_GRITv1.0"
}
MAX_DISTANCE_M = 100
SAMPLE_DISTANCE_M = 50

# Prepare GRIT: keep only needed columns
missing = [c for c in GRIT_COLS_TO_JOIN
           if c != "geometry" and c not in grit_raw.columns]
if missing:
    print(f"Columns not found in GRIT: {missing}")
else:
    print("All requested GRIT columns found.")

cols_keep = [c for c in GRIT_COLS_TO_JOIN if c in grit_raw.columns]
grit = grit_raw[cols_keep].copy()

All requested GRIT columns found.


In [9]:
# Function from "_2_1_1_line.py"
sword_2_1_1 = join_line_majority(
    sword= sword,
    grit= grit,
    cols_to_join = GRIT_COLS_TO_JOIN,
    rename_cols= GRIT_RENAME,
    max_distance_m = MAX_DISTANCE_M,
    sample_distance_m = SAMPLE_DISTANCE_M
)

# Convert to integer
for col in ["strahler_order_GRITv1.0", "global_id_GRITv1.0"]:
    if col in sword_2_1_1.columns:
        sword_2_1_1[col] = sword_2_1_1[col].astype("Int64")

print("\nNew columns added:")
new_cols = ["strahler_order_GRITv1.0", "global_id_GRITv1.0",
            "grit_majority_confidence"]
print(sword_2_1_1[["reach_id"] + new_cols].head(10))

SWORD reaches: 140
Sample points: 29681
Avg pts/reach: 212.0

Results:
  Matched   : 138 / 140
  Unmatched : 2
  Confidence distribution:
  count    138.000
mean       0.761
std        0.234
min        0.207
25%        0.553
50%        0.780
75%        1.000
max        1.000
Name: grit_majority_confidence, dtype: float64

New columns added:
      reach_id  strahler_order_GRITv1.0  global_id_GRITv1.0  \
0  46219400056                       16            30075767   
1  46219400041                       19            30075753   
2  46219400021                       23            30079274   
3  46219300051                       68            30075709   
4  46219300061                       65            30075696   
5  46219300091                       62            30075692   
6  46219100011                      113            30075917   
7  46219100021                      112            30075839   
8  46219100031                      112            30075839   
9  46219100041             

In [ ]:
# Quality check:
print("Majority confidence score [0-1]:")
print(sword_2_1_1["grit_majority_confidence"].describe().round(3))

print(f"Ambiguous reaches (confidence < 0.6 or NaN): {sword_2_1_1['grit_ambiguous'].sum()}")

print("\nReaches per strahler_order_GRITv1.0:")
print(sword_2_1_1["strahler_order_GRITv1.0"].value_counts().sort_index())

# Save
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_2_1_1.to_file(OUT_2_1_1, driver="GPKG")
print(f"\nSaved: {OUT_2_1_1}")

LATEST = OUT_2_1_1

Majority confidence score [0-1]:
count    138.000
mean       0.761
std        0.234
min        0.207
25%        0.553
50%        0.780
75%        1.000
max        1.000
Name: grit_majority_confidence, dtype: float64

Ambiguous reaches (confidence < 0.6): 45

Reaches per strahler_order_GRITv1.0:
strahler_order_GRITv1.0
1       8
2       3
3      10
4       5
5       5
       ..
107     2
108     2
110     1
112     2
113     1
Name: count, Length: 68, dtype: Int64

Saved: C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_2_1_1_grit.gpkg


In [24]:
# Interactive Map to check the joined strahler order information:
m = sword_2_1_1.explore(
    column="reach_id",
    cmap="turbo",
    tooltip=["reach_id", "river_name", "strahler_order_GRITv1.0",
             "strm_order", "grit_majority_confidence", "global_id_GRITv1.0"],
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_2_1_1)
webbrowser.open(OUT_MAP_2_1_1)
print(f"Map saved: {OUT_MAP_2_1_1}")

Map saved: C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\outputs\02_1_1_grit_map.html


Visual comparison between original SWORD reaches and to GRITv1.0 strahler order classified SWORD reaches.

In [31]:
# ################################################################
# ### VISUAL COMPARISON ###
# #########################

# # creating two plots
# fig, axes = plt.subplots(3, 1, figsize=(10, 20))

# # Reproject to metric CRS for display
# sword_plot = sword_2_1_1.to_crs("EPSG:32643")

# #Left map: SWORD column "strm_order"
# sword_plot.plot(
#     ax=axes[0],
#     column="reach_id",
#     cmap="Set1",
#     linewidth=0.8,
#     legend=True,
#     legend_kwds={"label": "SWORD strm_order", "shrink": 0.6}
# )
# axes[0].set_title("SWORD reaches\n(original)", fontsize=12)
# axes[0].axis("off")

# #Left map: SWORD column "strm_order"
# sword_plot.plot(
#     ax=axes[1],
#     column="strm_order",
#     cmap="Set1",
#     linewidth=0.8,
#     legend=True,
#     legend_kwds={"label": "SWORD strm_order", "shrink": 0.6}
# )
# axes[0].set_title("SWORD reaches\n(original)", fontsize=12)
# axes[0].axis("off")

# # Right map: the added GRIT strahler_order (column: "strahler_order_GRITv1.0")
# sword_plot.plot(
#     ax=axes[2],
#     column="strahler_order_GRITv1.0",
#     cmap="turbo",
#     linewidth=0.8,
#     legend=True,
#     legend_kwds={"label": "GRIT strahler_order", "shrink": 0.6}
# )
# axes[1].set_title("GRIT: strahler_order_GRITv1.0\n(joined via majority vote)", fontsize=12)
# axes[1].axis("off")

# plt.suptitle("Comparison: SWORD strm_order vs. GRIT strahler_order", fontsize=14)
# plt.tight_layout()

# # Save
# out_comparison = os.path.join(DATA_OUTPUT, "02_1_1_comparison_strm_vs_grit.png")
# plt.savefig(out_comparison, dpi=150, bbox_inches="tight")
# plt.show()
# print(f"Saved: {out_comparison}")

# which SWORD reaches share the same GRIT strahler order segment 

grouped = (sword_2_1_1
    .dropna(subset=["global_id_GRITv1.0"])
    .groupby("global_id_GRITv1.0")
    .agg(
        strahler_order = ("strahler_order_GRITv1.0", "first"),
        n_reaches= ("reach_id", "count"),
        reach_ids = ("reach_id", list),
        confidence_min = ("grit_majority_confidence", "min"),
        confidence_mean= ("grit_majority_confidence", "mean")
    )
    .sort_values("n_reaches", ascending=False)
    .reset_index()
)

print(f"Total GRIT segments (global_ids): {len(grouped)}")
print(f"SWORD reaches covered: {grouped['n_reaches'].sum()}")
print(f"\nSegments with only 1 reach: {(grouped['n_reaches'] == 1).sum()}")
print(f"Segments with 2-5 reaches: {grouped['n_reaches'].between(2,5).sum()}")
print(f"Segments with >5 reaches: {(grouped['n_reaches'] > 5).sum()}")

print("\n" + "-"*70)
print("GRIT segments with multiple SWORD reaches:")


for _, row in grouped[grouped["n_reaches"] > 1].iterrows():
    print(f"\nglobal_id : {int(row['global_id_GRITv1.0']):<10} "
          f"strahler: {int(row['strahler_order']):<5} "
          f"n_reaches: {int(row['n_reaches']):<5} "
          f"conf_mean: {row['confidence_mean']:.2f}")
    print(f"  reach_ids: {row['reach_ids']}")
print("\n" + "-"*70)

grouped[["global_id_GRITv1.0", "strahler_order",
         "n_reaches", "confidence_min", "confidence_mean"]]

Total GRIT segments (global_ids): 104
SWORD reaches covered: 138

Segments with only 1 reach: 76
Segments with 2-5 reaches: 28
Segments with >5 reaches: 0

----------------------------------------------------------------------
GRIT segments with multiple SWORD reaches:

global_id : 30075742   strahler: 24    n_reaches: 5     conf_mean: 1.00
  reach_ids: [46219600051, 46219600043, 46219600031, 46219600024, 46219600011]

global_id : 30075338   strahler: 5     n_reaches: 3     conf_mean: 0.93
  reach_ids: [46219200141, 46219200151, 46219200161]

global_id : 30075670   strahler: 22    n_reaches: 3     conf_mean: 0.74
  reach_ids: [46219900041, 46219900061, 46219900051]

global_id : 30075493   strahler: 6     n_reaches: 3     conf_mean: 0.77
  reach_ids: [46219800071, 46219800051, 46219800061]

global_id : 30075448   strahler: 27    n_reaches: 2     conf_mean: 0.59
  reach_ids: [46219200051, 46219200061]

global_id : 30075456   strahler: 2     n_reaches: 2     conf_mean: 0.75
  reach_ids: [

,global_id_GRITv1.0,strahler_order,n_reaches,confidence_min,confidence_mean
0,30075742,24,5,0.977778,0.995556
1,30075338,5,3,0.812808,0.934729
2,30075670,22,3,0.513393,0.735345
3,30075493,6,3,0.641129,0.770127
4,30075448,27,2,0.447154,0.594545
...,...,...,...,...,...
99,30079246,4,1,0.695652,0.695652
100,30079260,12,1,0.507692,0.507692
101,30079274,23,1,0.415459,0.415459
102,30084843,85,1,0.510870,0.510870


---
### 2.1.2 | Point Snap 

Snaps GDW barrier points to the nearest SWORD reach line. Only instream features are used. Snapped points are exported separately. The snap result (reach_id) is later used to flag reaches with upstream dams.

To-Do:
- [ ] Some of the original GDW points are located quite far from the feature, which is why the spatial deviation from the dam remains quite high even after snapping.

References:

Ward, B. (2019): "How to leverage Geopandas for faster snapping of points to lines". URL: https://medium.com/@brendan_ward/how-to-leverage-geopandas-for-faster-snapping-of-points-to-lines-6113c94e59aa

Lehner, B. et al.(2024): Global Dam Watch database version 1.0. URL: https://figshare.com/articles/dataset/Global_Dam_Watch_database_version_1_0/25988293

https://www.globaldamwatch.org/database

In [ ]:
# Reload SWORD from 2.1.1 output:
sword_2_1_1 = gpd.read_file(OUT_2_1_1)
print(f"SWORD reloaded: {len(sword_2_1_1)} reaches")

gdw = gpd.read_file(IN_GDW)
print(f"GDW loaded: {len(gdw)} features | CRS: {gdw.crs}")
print(f"GDW columns: {gdw.columns.tolist()}")

# Filtering for instream features only:
gdw = gdw[gdw["INSTREAM"].str.lower() == "instream"]
print(f"GDW instream: {len(gdw)} features after filter")

In [ ]:
# Configuration:
# The tolerance in meter describes the max snap distance
SNAP_TOLERANCE_M = 164

# NOTE: GDW points are exported separately, not merged into SWORD.
# The reach_id column links snapped points back to SWORD reaches.

In [ ]:
# Function from src file "_2_1_2_point.py"
snapped_gdw = snap_points_to_lines(
    points = gdw,
    lines = sword_2_1_1,
    tolerance_m = SNAP_TOLERANCE_M,
    line_id_col = "reach_id"
)

# Save snapped points (separate file -> not merged into SWORD)
snapped_gdw.to_file(OUT_2_1_2, driver="GPKG")
print(f"Saved: {OUT_2_1_2}")

LATEST = OUT_2_1_2

---
### 2.1.3 | Polygon 

...

---
---
## 2.2 | Raster Data

---
### 2.2.1 | Raster Join: STI glowabio

Extracts raster pixel values along each SWORD reach. Buffer per reach = (width/2) + offset to account for SWORD positional uncertainty (~50–200m offset from actual river).


Since the reaches can be offset by up to 200 meters from the actual river, a buffer is set around the reaches,depending on the reach width (column: "width"), to extract pixels from raster data. 

To-Do:

- [ ] I don't like the buffer solution because, with the current logic, it could end up extracting an unnecessary number of incorrect pixels. Maybe I should use a flow direction grid to better specify the direction of the buffer?
- [ ] In "raster_join.py" add to function "def extract_raster_values" that only pixels which overlap the buffer with a min. area of.... =< 50%? are included in the mean/median calculation.

https://glowabio.org/

https://hydrography.org/hydrography90m/hydrography90m_layers/


In [ ]:
# Reload from latest vector output
sword_current = gpd.read_file(OUT_2_1_1)
print(f"SWORD reloaded: {len(sword_current)} reaches")

sword_2_2_1 = sword_current.copy()

for col_name, raster_path in RASTER_JOINS:
    print(f"Raster: {raster_path}")
    print(f"File exists: {os.path.exists(raster_path)}")
    sword_2_2_1 = extract_raster_values(
        sword_2_2_1,
        raster_path,
        col_name,
        width_col="width"
    )

In [ ]:
# Inspecting results:
raster_cols = [c for name, _ in RASTER_JOINS
               for c in [f"{name}_mean", f"{name}_median"]]

print("Sample:")
print(sword_2_2_1[["reach_id", "width"] + raster_cols].head(10))

print("\nStatistics:")
print(sword_2_2_1[raster_cols].describe())

# Save
sword_2_2_1.to_file(OUT_2_2_1, driver="GPKG")
print(f"\nSaved: {OUT_2_2_1}")

LATEST = OUT_2_2_1

---
---
## 2.3 | FINAL OUTPUT

Combining all joined attributes into one final SWORD GeoDataFrame.
This is the input for Notebook 03 (classification).

In [ ]:
# Load the latest raster-joined dataset as base
sword_final = gpd.read_file(LATEST)

print(f"Final SWORD dataset:")
print(f"Reaches : {len(sword_final)}")
print(f"Columns : {sword_final.columns.tolist()}")

# sword_final.to_file(OUT_FINAL, driver="GPKG")
# print(f"\nSaved final output: {OUT_FINAL}")

In [ ]:
tooltip_cols = [c for c in [
    "reach_id", "river_name", "strahler_order_GRITv1.0",
    "grit_majority_confidence"
] if c in sword_final.columns]

m = sword_final.explore(
    column="strahler_order_GRITv1.0",
    cmap="RdYlBu",
    tooltip=tooltip_cols,
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_FINAL)
webbrowser.open(OUT_MAP_FINAL)
print(f"Map saved: {OUT_MAP_FINAL}")